# Etap 2 — wartości zdarzeń

Masa każdego odsypu liczona 9 wariantami (najbliższa próbka, mediana, średnia z okna 3 s × przerwa po końcu sygnału d = 0, 2, 3 s), masy zasypów dwoma sposobami i test półsyntetyczny (sztuczny odsyp o znanej masie na fragmentach bez zdarzeń). Ustalenia: `praca-inzynierska/zakres.md` §7 „Etap 2 — ustalenia”, plan `plans/dms-data-analysis/etap2-wartosci-zdarzen-plan.md`.

Dane: `data/results/<obiekt>/<id>/event_values.db` z `uv run python -m dms_analysis.events compute` (po `prepare`). **Wariant podstawowy:** mediana 3 s, d = 3 s.

Wyniki niezwalidowane, do przeglądu.

In [ ]:
from IPython.display import Markdown, display

from dms_analysis import style
from dms_analysis.data.catalog import load_catalog
from dms_analysis.events import browser, views

style.register()
results = views.load_all(load_catalog())
for warning in views.warnings(results):
    display(Markdown(f"<span style='color:#c0392b'><b>Uwaga:</b> {warning}</span>"))

## 1. Odsypy w okresach

Z analizy wypadają tylko odsypy, których okno (3 s przed, do 6 s po końcu) nachodzi na inne zdarzenie, lukę, granicę części okresu albo problem z katalogu (**okno**), oraz odsypy nachodzące na zasyp (**kolizja**, analizowane osobno później). Odsypy bez spadku wskazań zostają w analizie.

In [ ]:
views.summary_table(results)

## 2. Przeglądarka odsypu

Wybór okresu i doby, kolejność: po kolei w czasie, największa rozbieżność między wariantami, tylko odsypy bez spadku wskazań (w którymkolwiek wariancie), największa masa. Po prawej masy 9 wariantów: ramka = wariant podstawowy, kolor = różnica względem niego.

In [ ]:
browser.show(results)

## 3. Zestawienie mas odsypów

Masy w wariancie podstawowym, osobno dla każdego okresu, potem różnice pozostałych wariantów względem podstawowego (jak często i o ile wariant zmienia wynik).

In [ ]:
views.mass_histograms(results).show()

In [ ]:
views.variant_differences(results).show()

In [ ]:
views.no_drop_table(results)

## 4. Zasypy

Wszystkie zasypy są wpisane ręcznie (granice z marginesem). Masa = wskazanie po − wskazanie przed: z pojedynczych próbek i z mediany 3 s. Odsypy w trakcie zasypu zmniejszają masę liczoną z różnicy wskazań (ważne dla bilansu w etapie 3). Na wykresach: niebieskie pasmo = zasyp, szare pasma = odsypy.

In [ ]:
views.refills_table(results)

In [ ]:
for full_id, item in results.items():
    for figure in views.refill_figures(full_id, item):
        figure.show()

## 5. Test półsyntetyczny

Na fragment wskazań bez zdarzeń (co najmniej 60 s od prawdziwego zdarzenia, bez problemów z katalogu i luk) nakładamy rampę o masie i czasie trwania wylosowanych parami z prawdziwych odsypów wszystkich okresów; do masy dodany losowy ułamek działki (±10 kg), żeby znana masa nie leżała na siatce 20 kg jak prawdziwe odsypy; wskazania ze sztucznym odsypem zaokrąglone do działki jak w mierniku. Każdy wariant liczy masę, błąd = wynik − znana masa. Rampa nie ma drgań, więc test mierzy działanie metod na szum i dryf, nie na drgania w trakcie i po odsypie.

### 5.1. Granice dnia i nocy

Do wyboru granic: godziny pierwszego i ostatniego odsypu każdej doby. Okres `swietochlowice-1/0002-2026-09-28` ma czas szacowany (±2 h), więc pokazany osobnym kolorem. Po wyborze granice wpisujemy w `dms_analysis.events.synthetic` (`DAY_START_H`, `DAY_END_H`) i liczymy ponownie.

In [ ]:
views.first_last_hours(results).show()

### 5.2. Pokrycie i liczba prób

**Po co:** sprawdzenie, czy prób jest wystarczająco dużo i czy są rozłożone po całej dobie.

- **Pokrycie:** każdy punkt to start jednej próby (sztucznego odsypu) na osi godzin danej doby. Luki to miejsca zajęte przez prawdziwe zdarzenia z marginesem 60 s.
- **Zbieżność:** te same miary błędu liczone tylko z pierwszych 100, 250, 500 i 1000 prób każdej doby i pory. Jeśli linia jest płaska, kolejne próby nie zmieniają wyniku, czyli prób jest dość. Średni błąd ma stałą skalę ±10 kg, żeby wahania rzędu 1 kg (przypadkowe) nie wyglądały jak trend.
- **Grupy:** „z temperaturami” = okresy `swietochlowice-1/0004-2026-10-30` i `0005-2026-11-02` razem, „bez temperatur” = `0002-2026-09-28` i `0003-2026-10-21` razem (osobne wyniki zgodnie z ustaleniem z §7).

In [ ]:
for full_id, item in results.items():
    views.coverage_figure(full_id, item).show()

In [ ]:
views.convergence_figure(results).show()

### 5.3. Błąd wariantów

Rozkład błędu każdego wariantu, osobno okresy z temperaturami i bez; dzień i noc nałożone, oś pionowa = procent prób danej pory w przedziale 20 kg. Pod spodem tabela miar: odchylenie standardowe i 95. percentyl |błędu| (rozrzut), średni błąd (czy metoda systematycznie zawyża albo zaniża — średnia z tysięcy prób, więc ułamkowa).

In [ ]:
for group_name in views.GROUPS:
    views.error_histograms(results, group_name).show()

In [ ]:
views.error_table(results)

### 5.4. Błąd wg godziny i doby (wariant podstawowy)

In [ ]:
views.error_by_hour(results).show()

In [ ]:
views.error_by_day(results).show()

### 5.5. Największe błędy

Sześć prób z największym błędem w każdej grupie: szary = oryginalne wskazania, niebieski = ze sztucznym odsypem, szare pasmo = sztuczny odsyp.

In [ ]:
for group_name in views.GROUPS:
    views.worst_trials(results, group_name).show()